In [ ]:
!pip install neo4j
!pip install neo4j sentence-transformers
!pip install python-dotenv
!pip install google-generativeai



  Using cached neo4j-6.1.0-py3-none-any.whl.metadata (5.3 kB)
Using cached neo4j-6.1.0-py3-none-any.whl (325 kB)
   ---------------------------------------- 0.0/510.5 kB ? eta -:--:--
    --------------------------------------- 10.2/510.5 kB ? eta -:--:--
   ----- --------------------------------- 71.7/510.5 kB 975.2 kB/s eta 0:00:01
   ---------------- ----------------------- 204.8/510.5 kB 2.1 MB/s eta 0:00:01
   ------------------------------------- -- 481.3/510.5 kB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 510.5/510.5 kB 2.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\edex\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ---------------------------------------- 57.7/57.7 kB 3.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB 1.0 MB/s eta 0:00:00
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/570.8 kB ? eta -:--:--
   ---------- ----------------------------- 153.6/570.8 kB 4.6 MB/s eta 0:00:01
   --------------- ------------------------ 225.3/570.8 kB 4.6 MB/s eta 0:00:01
   --------------------- ------------------ 307.2/570.8 kB 2.4 MB/s eta 0:00:01
   ----------------------------- ---------- 419.8/570.8 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 570.8/570.8 kB 2.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/642.6 kB ? eta -:--:--
   --------------- ------------------------ 256.0/642.6 kB 5.2 MB/s eta 0:00:01
   ---------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\edex\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [24]:
import os
import json
from pathlib import Path
from neo4j import GraphDatabase
from dotenv import load_dotenv
load_dotenv()


# --- Connection ---
# These now pull from your .env file automatically
URI = os.environ.get("NEO4J_URI")
USER = "92fb1806"
PASSWORD = "tu_ZeJEs2_HR8Ql8QAzvu8lQkVdzK14m4WNOhEktVqY"

# --- Limits ---
MAX_ENTITY_NODES = 49_000
MAX_RELATIONS = 170_000
BATCH_SIZE = 500

CLEAR_DATABASE = True
CREATE_VECTOR_INDEX = True
VECTOR_DIMENSIONS = 384


In [25]:
def extracted_jsonl(name: str) -> Path:
    cwd = Path.cwd()
    for base in (cwd, cwd / "scripts", cwd.parent):
        p = base / "data_clean" / "extracted" / name
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(f"Could not find data_clean/extracted/{name}")

def setup_database(tx, create_vector_index: bool = False):
    tx.run("CREATE CONSTRAINT entity_id_unique IF NOT EXISTS FOR (e:Entity) REQUIRE e.id IS UNIQUE")
    if create_vector_index:
        tx.run(f"""
            CREATE VECTOR INDEX entity_embeddings IF NOT EXISTS
            FOR (e:Entity) ON (e.embedding)
            OPTIONS {{indexConfig: {{
             `vector.dimensions`: {VECTOR_DIMENSIONS},
             `vector.similarity_function`: 'cosine'
            }}}}
        """)

def clear_all(tx):
    tx.run("MATCH (n) DETACH DELETE n")

def load_entities_batch(tx, batch):
    q = """
    UNWIND $batch AS item
    WITH item
    WHERE item.id IS NOT NULL AND item.id <> ''
    MERGE (e:Entity {id: item.id})
    SET e.text = item.id,
        e.label = item.label,
        e.embedding = item.embedding  // NEW: Store calculated vector
    """
    tx.run(q, batch=batch)

def load_relations_batch(tx, batch):
    q = """
    UNWIND $batch AS rel
    WITH rel
    WHERE rel.head_text IS NOT NULL AND rel.tail_text IS NOT NULL AND rel.label IS NOT NULL
    MATCH (src:Entity {id: rel.head_text})
    MATCH (tgt:Entity {id: rel.tail_text})
    MERGE (src)-[r:REL {label: rel.label}]->(tgt)
    SET r.score = rel.score
    """
    tx.run(q, batch=batch)

def build_allowed_entities(entities_path: Path, max_unique: int) -> dict[str, dict]:
    out: dict[str, dict] = {}
    with open(entities_path, "r", encoding="utf-8") as f:
        for line in f:
            if len(out) >= max_unique: break
            o = json.loads(line.strip())
            t = (o.get("text") or "").strip()
            if t and t not in out:
                out[t] = {"id": t, "label": o.get("label")}
    return out

def write_entities(driver, entities: dict[str, dict], batch_size: int) -> int:
    # --- NEW: Embedding Generation ---
    from sentence_transformers import SentenceTransformer
    print("Initializing embedding model (all-MiniLM-L6-v2)...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    items = list(entities.values())
    n = len(items)
    with driver.session() as session:
        for i in range(0, n, batch_size):
            batch = items[i : i + batch_size]
            texts = [item["id"] for item in batch]
            
            # Generate embeddings for the batch
            embeddings = model.encode(texts, show_progress_bar=False).tolist()
            for item, emb in zip(batch, embeddings):
                item["embedding"] = emb
                
            session.execute_write(load_entities_batch, batch)
            if (i + batch_size) % 5000 == 0 or i + batch_size >= n:
                print(f"  Progress: {min(i + batch_size, n):,} / {n:,} entities embedded and uploaded.")
    return n

def write_relations_capped(driver, relations_path: Path, allowed: set[str], max_rels: int, batch_size: int) -> int:
    written = 0
    batch = []
    with driver.session() as session:
        with open(relations_path, "r", encoding="utf-8") as f:
            for line in f:
                if written >= max_rels: break
                r = json.loads(line.strip())
                ht = (r.get("head") or {}).get("text", "").strip()
                tt = (r.get("tail") or {}).get("text", "").strip()
                lab = r.get("label")
                if ht in allowed and tt in allowed and lab:
                    batch.append({"head_text": ht, "tail_text": tt, "label": lab, "score": r.get("score")})
                    if len(batch) >= batch_size:
                        session.execute_write(load_relations_batch, batch)
                        written += len(batch)
                        batch = []
        if batch:
            session.execute_write(load_relations_batch, batch)
            written += len(batch)
    return written

def run_full_load():
    entities_path = extracted_jsonl("entities.jsonl")
    relations_path = extracted_jsonl("relations.jsonl")
    
    with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as driver:
        driver.verify_connectivity()
        with driver.session() as session:
            if CLEAR_DATABASE:
                print("Clearing graph...")
                session.execute_write(clear_all)
            session.execute_write(lambda tx: setup_database(tx, CREATE_VECTOR_INDEX))

        entities = build_allowed_entities(entities_path, MAX_ENTITY_NODES)
        allowed = set(entities.keys())
        
        print(f"Uploading {len(allowed):,} entities with embeddings...")
        n_ent = write_entities(driver, entities, BATCH_SIZE)

        print(f"Uploading relations (cap {MAX_RELATIONS:,})...")
        n_rel = write_relations_capped(driver, relations_path, allowed, MAX_RELATIONS, BATCH_SIZE)
        print(f"Done. {n_ent:,} nodes and {n_rel:,} relations written.")

run_full_load()


Clearing graph...
Uploading 3,442 entities with embeddings...


C:\Users\edex\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing embedding model (all-MiniLM-L6-v2)...


C:\Users\edex\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\edex\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading wei

  Progress: 3,442 / 3,442 entities embedded and uploaded.
Uploading relations (cap 170,000)...
Done. 3,442 nodes and 170,000 relations written.


In [26]:
!pip install google-generativeai


     ---------------------------------------- 0.0/90.6 kB ? eta -:--:--
     ------------------------------- -------- 71.7/90.6 kB 2.0 MB/s eta 0:00:01
     ---------------------------------------- 90.6/90.6 kB 1.0 MB/s eta 0:00:00
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.78.0-py3-none-any.whl.metadata (1.3 kB)
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/155.1 kB ? eta -:--:--
   ---------------------------------------  153.6/155.1 kB 9.6 MB/s eta 0:00:01
   ---------------------------------------  153.6/155.1 kB 9.6 MB/s eta 0:00:01
   ---------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\edex\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
